In [ ]:
N_TEST = 1500
N_TRAIN = 4000

SEED = 4

SYNTH_FOLDER = "synth/30/"
POP_DATASET = "population.csv"
FEATURE_MAP = "sps/toy_features.json"

In [ ]:
import sys

PROJECT_ROOT = '../../'
SRC = f"{PROJECT_ROOT}/src"
if PROJECT_ROOT not in sys.path:
  sys.path.append(PROJECT_ROOT)
if SRC not in sys.path:
  sys.path.append(SRC)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import json
from config import Config
from notebooks.analysis_utils import print_table_1

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

In [ ]:
total_population = pd.read_csv(f"{PROJECT_ROOT}/{Config.DATA_DIR}/{SYNTH_FOLDER}/{POP_DATASET}")
with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

continuous_features = [f['name'] for f in feature_map['X'] if f['type'] == "continuous"]
categorical_features = [f['name'] for f in feature_map['X'] if f['type'] != "continuous"]
target_col = feature_map['target']['name']
sens_col = feature_map['sens'][0]['name']

# Population summary

In [ ]:
print_table_1(total_population, continuous_features, categorical_features + [target_col], groupby=sens_col)

# TRAINING / TEST split

In [ ]:
from sklearn.model_selection import train_test_split

strata = total_population[[target_col, sens_col]]

train_df, test_df = train_test_split(
  total_population,
  test_size=N_TEST,
  train_size=N_TRAIN,
  stratify=strata,
  random_state=SEED
)

In [ ]:
print(f"CEVAE-HE Training Set Size: {len(train_df)}")
print(f"CEVAE-HE Test Set Size: {len(test_df)}")

print("="*60)
print("Training Set Sens. Distribution by Target Outcome:")
print("="*60)
print(pd.crosstab(train_df[sens_col], train_df[target_col], margins=True))
print('\n')
print((pd.crosstab(train_df[sens_col], train_df[target_col], normalize="all", margins=True)*100).round(2))
print('\n')
print("="*60)
print("Test Set Sens. Distribution by Target Outcome:")
print("="*60)
print(pd.crosstab(test_df[sens_col], test_df[target_col], margins=True))
print('\n')
print((pd.crosstab(test_df[sens_col], test_df[target_col], normalize="all",margins=True)*100).round(2))

# Scaling

In [ ]:
from notebooks.processing_utils import StrataAwareRobustScaler

strata_scaler = StrataAwareRobustScaler(strata_col=sens_col, continuous_cols=continuous_features)
scaled_train_df = strata_scaler.fit_transform(train_df)
scaled_test_df = strata_scaler.transform(test_df)

## Training table 1

In [ ]:
print_table_1(scaled_train_df, continuous_features, categorical_features + [target_col], groupby=sens_col)

## Test table 1

In [ ]:
print_table_1(scaled_test_df, continuous_features, categorical_features + [target_col], groupby=sens_col)

## Save splits

In [ ]:
scaled_train_df.to_csv(f"{PROJECT_ROOT}/{Config.DATA_DIR}/{SYNTH_FOLDER}/train.csv")
scaled_test_df.to_csv(f"{PROJECT_ROOT}/{Config.DATA_DIR}/{SYNTH_FOLDER}/test.csv")